In [0]:
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

# Explicit schema for GH Archive events
# payload stored as STRING (raw JSON) to avoid schema merge conflicts across event types
GHARCHIVE_SCHEMA = StructType([
    StructField("id", StringType()),
    StructField("type", StringType()),
    StructField("actor", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
        StructField("display_login", StringType()),
        StructField("gravatar_id", StringType()),
        StructField("url", StringType()),
        StructField("avatar_url", StringType()),
    ])),
    StructField("repo", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
        StructField("url", StringType()),
    ])),
    StructField("payload", StringType()),
    StructField("public", BooleanType()),
    StructField("created_at", StringType()),
    StructField("org", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
        StructField("gravatar_id", StringType()),
        StructField("url", StringType()),
        StructField("avatar_url", StringType()),
    ])),
])

# Widget: specific filename (no wildcard fallback)
dbutils.widgets.text("filename", "", "Filename (e.g. 2026-03-25-3.json.gz)")

# Get filename from upstream ingest task, fall back to widget
try:
    filename = dbutils.jobs.taskValues.get(taskKey="ingest", key="filename")
    print(f"Got filename from ingest task: {filename}")
except Exception as e:
    filename = dbutils.widgets.get("filename")
    if not filename:
        raise ValueError("No filename provided. Set the widget or run via job pipeline.")
    print(f"Task value not available ({e}), using widget value: {filename}")

VOLUME_PATH = f"/Volumes/gharchive_dev/raw/files/{filename}"
BRONZE_TABLE = "gharchive_dev.v1_pyspark.gharchive_bronze"

print(f"Reading from: {VOLUME_PATH}")

# Read with explicit schema + rescued data column for any unexpected fields
df_raw = (
    spark.read
    .format("json")
    .schema(GHARCHIVE_SCHEMA)
    .option("multiLine", "false")
    .option("rescuedDataColumn", "_rescued_data")
    .load(VOLUME_PATH)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

# Filter out files already ingested (deduplication)
if spark.catalog.tableExists(BRONZE_TABLE):
    existing_files = (
        spark.read.table(BRONZE_TABLE)
        .select("_source_file")
        .distinct()
    )
    df_bronze = df_raw.join(existing_files, on="_source_file", how="left_anti")
    new_file_count = df_bronze.select("_source_file").distinct().count()
    print(f"New files to ingest: {new_file_count}")
else:
    df_bronze = df_raw
    print("Table does not exist yet. Ingesting all files.")

# Append only new data to Delta table
if df_bronze.head(1):
    df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(BRONZE_TABLE)
    print(f"Bronze table appended: {BRONZE_TABLE}")
else:
    print("No new files to ingest. Skipping write.")

print(f"Total row count: {spark.read.table(BRONZE_TABLE).count()}")